In [1]:
%%capture
%pip install accelerate peft bitsandbytes transformers trl

In [ ]:
# # Login to Hugging Face
from huggingface_hub import login
login("token")


In [3]:
import wandb

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

wb_token = user_secrets.get_secret("wandb")

# Solo necesitas `wandb.login()` si no has iniciado sesión antes
wandb.login(key=wb_token)

# Inicia un nuevo run en wandb (esto debe hacerse cada vez que ejecutas el proyecto)
run = wandb.init(
    project="Fine-tune-Emotion-Detector-Llama-2-7b-ft-instruct-es-training-v4", 
    job_type="training", 
    anonymous="allow"
)

print("Run iniciado correctamente.")


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: joseph-rios (joseph-rios-unl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Run iniciado correctamente.


In [4]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"  # Suppress TensorFlow logs
import warnings
warnings.filterwarnings("ignore")

In [3]:
import os
import torch
from datasets import load_dataset, DatasetDict
from collections import Counter
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig
from trl import SFTTrainer
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import re

In [6]:
# Model from Hugging Face hub
base_model = "clibrain/Llama-2-7b-ft-instruct-es"

# New instruction dataset
emotion_dataset = "Joseph7D/prompt-emotion-dataset-v2"

dataset = load_dataset(emotion_dataset)
train_dataset = dataset['train']        # 80% 
eval_dataset  = dataset['validation']   # 10% 
test_dataset  = dataset['test']         # 10% 

# Fine-tuned model
new_model = "llama-2-7b-emotion-detector-v4"

README.md:   0%|          | 0.00/671 [00:00<?, ?B/s]

(…)-00000-of-00001-0c67affb9210ad40.parquet:   0%|          | 0.00/2.38M [00:00<?, ?B/s]

(…)-00000-of-00001-4f62525995947485.parquet:   0%|          | 0.00/300k [00:00<?, ?B/s]

(…)-00000-of-00001-99d9ca9cf25d3c0d.parquet:   0%|          | 0.00/300k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26934 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3366 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3372 [00:00<?, ? examples/s]

In [1]:
# Model from Hugging Face hub
base_model = "clibrain/Llama-2-7b-ft-instruct-es"

In [7]:
compute_dtype = getattr(torch, "float16")

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=False,
)

In [8]:
# Load base model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quant_config,
    device_map={"": 0}
)
model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

pytorch_model.bin.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

pytorch_model-00002-of-00002.bin:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

pytorch_model-00001-of-00002.bin:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [9]:
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


In [10]:
import bitsandbytes as bnb

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)


modules = find_all_linear_names(model)
modules

['v_proj', 'q_proj', 'down_proj', 'o_proj', 'k_proj', 'gate_proj', 'up_proj']

In [ ]:
# Load LoRA configuration
peft_args = LoraConfig(
    lora_alpha=32,
    lora_dropout=0.05,
    r=16,
    bias="lora_only",
    task_type="CAUSAL_LM",
    target_modules=modules,
)


In [ ]:
# Set training parameters
output_dir="/kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-training4"

training_params = TrainingArguments(
    output_dir=output_dir,                    # directory to save and repository id
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    save_steps=2000,             # guardar cada 1000 pasos
    save_total_limit=1,          # mantener solo el checkpoint más reciente
    logging_steps=100,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to="wandb",                  # report metrics to w&b
    disable_tqdm=False,          
    eval_strategy="steps",
    eval_steps=2000,
)


In [13]:
# Set supervised fine-tuning parameters
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=peft_args,
    args=training_params,
)

tokenizer_config.json:   0%|          | 0.00/725 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/437 [00:00<?, ?B/s]

Converting train dataset to ChatML:   0%|          | 0/26934 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/26934 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/26934 [00:00<?, ? examples/s]

...

Converting eval dataset to ChatML:   0%|          | 0/3372 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/3372 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/3372 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/3372 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


No se busca una clasificacion explicita, sino mas bien una generaricon condicionada

El warning de label_names:

"No label_names provided for model class PeftModelForCausalLM..."

puede ser ignorado, porque el modelo no está haciendo clasificación explícita, sino generación condicionada.

💡 Trainer espera clasificación si se usa compute_metrics, pero aquí es generación. Así que se puede usar un Trainer personalizado o Seq2SeqTrainer, o incluso un bucle de entrenamiento.

In [14]:
# Train model
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss
2000,0.659100,0.726392


TrainOutput(global_step=3366, training_loss=0.6805769749787048, metrics={'train_runtime': 27692.9965, 'train_samples_per_second': 0.973, 'train_steps_per_second': 0.122, 'total_flos': 1.059821937462313e+17, 'train_loss': 0.6805769749787048})

In [18]:
# # Directorio donde se guardará el modelo
output_dir = "/kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-training4"

# Guarda sólo los pesos del modelo y el tokenizer
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
# !zip -r /kaggle/working/modelo-ajustado.zip /kaggle/working/llama-2-7b-emotion-detector-v2

('/kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/tokenizer_config.json',
 '/kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/special_tokens_map.json',
 '/kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/tokenizer.json')

In [25]:
logging.set_verbosity(logging.CRITICAL)

# Suprime mensajes de advertencia
logging.set_verbosity(logging.CRITICAL)

# Define el prompt con el formato adecuado
prompt = """Clasifica el siguiente texto en una de estas emociones: ira, disgusto, tristeza, alegría, miedo o neutral. Responde únicamente con la emoción correspondiente.

Texto: "Me indigna ver cómo algunas personas menosprecian los derechos humanos"
"""

# Genera el resultado usando el pipeline
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200)
result = pipe(f"<s>[INST] {prompt} [/INST]")

# Muestra el resultado
print(result[0]['generated_text'])

<s>[INST] Clasifica el siguiente texto en una de estas emociones: ira, disgusto, tristeza, alegría, miedo o neutral. Responde únicamente con la emoción correspondiente.

Texto: "Me indigna ver cómo algunas personas menosprecian los derechos humanos"
 [/INST] ira 


In [33]:
!zip -r /kaggle/working/llama-emotions-output.zip /kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-training4


  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/ (stored 0%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/tokenizer.json (deflated 85%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/adapter_model.safetensors

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


 (deflated 7%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/adapter_config.json (deflated 55%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/special_tokens_map.json (deflated 48%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/README.md (deflated 66%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/checkpoint-3366/ (stored 0%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/checkpoint-3366/tokenizer.json (deflated 85%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/checkpoint-3366/adapter_model.safetensors (deflated 7%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/checkpoint-3366/training_args.bin (deflated 51%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/checkpoint-3366/rng_state.pth (deflated 25%)
  adding: kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-testing4/checkpoint-3366/adapte

In [4]:
# Reload model in FP16 and merge it with LoRA weights
import gc
from peft import PeftModel  # Import PeftModel
!pip install safetensors
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()  # Clear the GPU cache

# Specify offload folder
offload_folder = "./offload"  # Or any directory you prefer
os.makedirs(offload_folder, exist_ok=True)  # Create if it doesn't exist

# Configure BitsAndBytes for 8-bit loading with CPU offload
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True  # Enable CPU offload for 32-bit parts
)

# Load the model using the BitsAndBytes config
load_model = AutoModelForCausalLM.from_pretrained(
    base_model,
    low_cpu_mem_usage=True,
    return_dict=True,
    torch_dtype=torch.float16,
    device_map="auto",  # Load on CPU or use device_map="auto"
    offload_folder=offload_folder,  # Specify offload folder
    quantization_config=bnb_config,  # Use the BitsAndBytes config
)

# Pass the offload_folder to PeftModel.from_pretrained
model = PeftModel.from_pretrained(load_model, "/kaggle/working/Llama-2-7b-ft-instruct-es-fine-tuned-training4", offload_folder=offload_folder)
model = model.merge_and_unload()

# Reload tokenizer to save it
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.add_special_tokens({"pad_token": "[PAD]"})
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Delete the load_model to free up memory
del load_model
gc.collect()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`. This was detected when initializing the generation config instance, which means the corresponding file may hold incorrect parameterization and should be fixed.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/generation/configuratio

169

In [11]:
# Fine-tuned model
# emotion_dataset = "Joseph7D/prompt-emotion-dataset-v2"

# new_model = "Joseph7D/llama-2-7b-emotion-detector-v4"
# Update generation config
model.generation_config.temperature = None  # Unset temperature
model.generation_config.top_p = None  # Unset top_p

# Reload tokenizer to save it
model.push_to_hub("Joseph7D/llama-2-7b-emotion-detector-v4", use_temp_dir=True)
tokenizer.push_to_hub("Joseph7D/llama-2-7b-emotion-detector-v4", use_temp_dir=True)


model-00002-of-00002.safetensors:   0%|          | 0.00/2.02G [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.99G [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/Joseph7D/llama-2-7b-emotion-detector-v4/commit/f53fe004666920c5a0674962d66f8ab33a0e1f89', commit_message='Upload tokenizer', commit_description='', oid='f53fe004666920c5a0674962d66f8ab33a0e1f89', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Joseph7D/llama-2-7b-emotion-detector-v4', endpoint='https://huggingface.co', repo_type='model', repo_id='Joseph7D/llama-2-7b-emotion-detector-v4'), pr_revision=None, pr_num=None)